In [ ]:
import json
import numpy as np

import os
from pathlib import Path
from train_utils import build_train_val_tables, run_trial, save_hyperparam_results
from hypothesis_config import HYPOTHESES

In [5]:


PROJECT_ROOT = Path().resolve().parent

DATASET_ROOT = Path(
    os.getenv("EWS_DATASET_PATH", PROJECT_ROOT / "data" / "EWS-Dataset")
)

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR   = DATASET_ROOT / "validation"
TEST_DIR  = DATASET_ROOT / "test"

X_train, y_train, val_img_paths, val_mask_paths = build_train_val_tables(TRAIN_DIR, VAL_DIR)

Train: 142 images
Val:   24 images
Building training table...
X_train: (1396180, 13)  y_train: (1396180,)


In [ ]:
all_results = []

for trial in HYPOTHESES:
    print(f"Running: {trial['hypothesis']} ...")
    metrics = run_trial(
        trial["params"],
        X_train, y_train,           # from your Section 3
        val_img_paths, val_mask_paths,
    )
    all_results.append({
        "hypothesis":   trial["hypothesis"],
        "rationale":    trial["rationale"],
        "params":       trial["params"],
        **metrics,
    })
    print(f"  IoU={metrics['mean_iou']:.4f}  ±{metrics['std_iou']:.4f}  "
          f"train={metrics['train_time_s']:.1f}s")

print("\nDone.")

Running: baseline ...
  IoU=0.8362  ±0.1321  train=131.3s
Running: depth_control_mild ...
  IoU=0.8367  ±0.1317  train=105.9s
Running: depth_control_aggressive ...
  IoU=0.8393  ±0.1302  train=71.8s
Running: smooth_boundary_small_leaf ...
  IoU=0.8375  ±0.1312  train=98.0s
Running: smooth_boundary_large_leaf ...
  IoU=0.8377  ±0.1311  train=92.7s
Running: more_features_per_split ...
  IoU=0.8360  ±0.1320  train=189.0s
Running: all_features_per_split ...
  IoU=0.8355  ±0.1321  train=457.3s
Running: more_trees ...
  IoU=0.8364  ±0.1316  train=220.6s
Running: combined_depth_and_leaf ...
  IoU=0.8376  ±0.1311  train=103.1s

Done.


In [7]:
best = max(all_results, key=lambda x: x["mean_iou"])

output = {
    "best_hypothesis": best["hypothesis"],
    "best_params":     best["params"],
    "best_iou":        best["mean_iou"],
    "rationale":       best["rationale"],
    "n_trials":        len(all_results),
    "all_trials":      all_results,
}

with open("rf_hyperparam_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Best: {best['hypothesis']}  IoU={best['mean_iou']:.4f}")
print("Saved: rf_hyperparam_results.json")

Best: depth_control_aggressive  IoU=0.8393
Saved: rf_hyperparam_results.json
